In [1]:
import pandas as pd

df = pd.read_excel("units/toat_platform.xlsx", skiprows=2)

In [2]:
folder_path = "wishlists/Annapurna Interactive Historical Revenues - releases_dates.csv"
dates  = pd.read_csv(folder_path)
dates = dates.dropna(subset='pc_release_date')
dates['pc_release_date'] = pd.to_datetime(dates['pc_release_date'])
release_date_dict = dates.groupby('product')['pc_release_date'].min().to_dict()

In [3]:
release_date_dict.keys()

dict_keys(['A Memoir Blue', 'Ashen', 'BattleSage', 'Cocoon', 'Donut County', 'Due Process', 'Faraway', 'Flock', 'Florence', 'Flower', 'Gorogoa', 'Hindsight', 'Hohokum', 'I Am Dead', 'If Found...', 'Journey', 'Kentucky Route Zero: TV Edition', 'Last Stop', 'Lorelei and the Laser Eyes', 'Lushfoil Photography Sim', 'Maquette', 'Mixtape', 'Morsels', 'Mundaun', 'Neon White', 'Open Roads', 'Outer Wilds', 'Sayonara Wild Hearts', 'Skin Deep', 'Solar Ash', 'Storyteller', 'Stray', 'Telling Lies', 'The Artful Escape', 'The Lost Wild', 'The Pathless', 'The Unfinished Swan', 'Thirsty Suitors', 'Twelve Minutes', 'Wanderstop', 'Wattam', 'What Remains of Edith Finch', 'Wheel World', 'to a T'])

In [4]:
TITLE = 'to a T'

In [5]:
import glob
import os

# Define folder path
folder_path = "/Users/dougs/Documents/GitHub/IndieBI-Sales-EDA/units/"

# Get all .xlsx files in the folder
files = glob.glob(f"{folder_path}/*.csv")

# Read all files into DataFrames and add filename column
dfs = [pd.read_csv(file).assign(Source=os.path.basename(file)) for file in files]
av_df = pd.concat(dfs, ignore_index=True)
av_df['platform'] = av_df['platform'].str.title()
av_df['platform'] = av_df['platform'].replace({'Playstation': 'PlayStation'})
av_df['platform'] = av_df['platform'].replace({'Xbox': 'Microsoft'})


In [6]:
curve_df = av_df.pivot(columns='platform', index='Weeks_from_Release', values="daily_delta").reset_index()

In [7]:
platform_list = list(df.portal.unique())

In [8]:
df = df.pivot(columns='portal', index='Day', values="Cumulative Selected Measure").reset_index()

In [9]:
df['release_date'] = release_date_dict.get(TITLE)
df['release_date'] = pd.to_datetime(df['release_date'])

In [10]:
df["DAR"] = (df["Day"] - df["release_date"]).dt.days
df["Weeks_from_Release"] = (df["DAR"] // 7) + 1  # Week 1 starts at 0-7 days


In [11]:
#df = df[['Weeks_from_Release']+platform_list]

In [12]:
actual_df = df.sort_values(by='DAR').drop_duplicates("Weeks_from_Release", keep='last')

In [13]:
actual_df = actual_df.drop(["Day",'release_date'], axis=1)

In [14]:
curve_df

platform,Weeks_from_Release,Apple,Epic,Gog,Google,Humble,Microsoft,Nintendo,PlayStation,Steam
0,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,1.023745,0.649729,0.267076,0.620922,0.559854,0.796915,1.101720,0.487674,0.388850
2,3,0.428988,1.043252,1.529236,0.229504,0.235810,0.363660,0.299089,0.434923,0.289576
3,4,0.349177,0.245943,0.127330,0.249465,0.043367,0.282220,0.132276,0.219304,0.250393
4,5,0.208720,0.136560,0.022225,0.184109,0.016351,1.957906,0.086132,0.232965,0.179397
...,...,...,...,...,...,...,...,...,...,...
151,152,NaN,0.004309,0.001176,0.002021,0.000443,0.001287,0.007436,0.004749,0.002553
152,153,NaN,0.002406,0.001029,0.008588,0.017499,0.001057,0.001912,0.005187,0.003941
153,154,NaN,0.006224,0.001299,0.010736,0.032510,0.001122,0.003941,0.004189,0.003873
154,155,NaN,0.014870,0.004310,0.002687,0.032969,0.001209,0.005332,0.001961,0.004511


In [15]:
import pandas as pd
import numpy as np
actual_df = actual_df.set_index('Weeks_from_Release')
curve_df = curve_df.set_index('Weeks_from_Release')


# 2. Get the last actual week and values
last_week = actual_df.index.max()
last_values = actual_df.loc[last_week].copy()

# 3. Create a DataFrame to hold projections
projected_df = pd.DataFrame()

# 4. Iteratively apply the growth deltas week by week
current_values = last_values.copy()

for week in range(last_week + 1, curve_df.index.max() + 1):
    if week not in curve_df.index:
        continue
    deltas = curve_df.loc[week].fillna(0)  # Default to 0 growth if missing
    current_values = current_values * (1 + deltas)
    current_values.name = week
    current_values = np.ceil(current_values.fillna(0)).astype(int)
    projected_df = pd.concat([projected_df, current_values.to_frame().T])

# 5. Combine with actuals
full_df = pd.concat([actual_df, projected_df])

# 6. Reset index if needed
full_df = full_df.reset_index()

In [16]:
full_df = full_df.drop("index", axis=1)

In [17]:
filtered_df = full_df[full_df.index < 52]
filtered_df

,Epic,Microsoft,PlayStation,Steam,DAR,Apple,Gog,Google,Humble,Nintendo
0,7,556,1742,2882,6,NaN,NaN,NaN,NaN,NaN
1,12,1000,2592,4003,0,0.0,0.0,0.0,0.0,0.0
2,25,1364,3720,5163,0,0.0,0.0,0.0,0.0,0.0
3,32,1749,4536,6456,0,0.0,0.0,0.0,0.0,0.0
4,37,5174,5593,7615,0,0.0,0.0,0.0,0.0,0.0
5,54,7097,6757,8300,0,0.0,0.0,0.0,0.0,0.0
6,59,9769,7680,8704,0,0.0,0.0,0.0,0.0,0.0
7,63,17792,8466,9194,0,0.0,0.0,0.0,0.0,0.0
8,68,20320,9191,9720,0,0.0,0.0,0.0,0.0,0.0
9,71,21353,9663,10405,0,0.0,0.0,0.0,0.0,0.0


In [18]:
full_df.loc[51]

Epic              286.0
Microsoft      220430.0
PlayStation     23575.0
Steam           27282.0
DAR                 0.0
Apple               0.0
Gog                 0.0
Google              0.0
Humble              0.0
Nintendo            0.0
Name: 51, dtype: float64

In [19]:
full_df.loc[51].sum()

271573.0

In [20]:
full_df.loc[103].sum()

594015.0

In [21]:
full_df.loc[155].sum()

930638.0

In [22]:
curve_df.query("Weeks_from_Release <=52")

platform,Apple,Epic,Gog,Google,Humble,Microsoft,Nintendo,PlayStation,Steam
Weeks_from_Release,,,,,,,,,
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.023745,0.649729,0.267076,0.620922,0.559854,0.796915,1.101720,0.487674,0.388850
3,0.428988,1.043252,1.529236,0.229504,0.235810,0.363660,0.299089,0.434923,0.289576
4,0.349177,0.245943,0.127330,0.249465,0.043367,0.282220,0.132276,0.219304,0.250393
5,0.208720,0.136560,0.022225,0.184109,0.016351,1.957906,0.086132,0.232965,0.179397
6,0.205037,0.435918,0.048797,0.120952,0.039756,0.371511,0.072239,0.208107,0.089845
7,0.200992,0.091931,0.037755,0.109518,0.174717,0.376466,0.042954,0.136509,0.048627
8,0.142654,0.062639,0.015608,0.106706,0.065560,0.821238,0.062439,0.102260,0.056229
9,0.112479,0.064807,0.010120,0.088285,0.039808,0.142072,0.062956,0.085522,0.057114
